# Reproduce Table 2 — Hierarchical Graph Benchmarks

**Paper:** Hyperbolic and Riemannian Geometry as Inductive Biases for Deep Neural Network Architecture Design  
**Section:** 7.1  
**Prerequisite:** All experiments in `scripts/reproduce/run_wordnet.sh` and `run_ogbn.sh` must be complete.

This notebook loads `outputs/*/results.json` for all model–seed combinations,
computes mean ± std over seeds 0–9, runs Wilcoxon signed-rank tests vs GeoNet,
and reproduces Table 2 exactly as reported in the paper.

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

# ── Configuration ─────────────────────────────────────────────────────────────
OUTPUT_ROOT = Path('outputs')
SEEDS       = list(range(10))
MODELS      = ['geonet', 'gcn', 'gatv2', 'graphsage_v2', 'hgcn_plus', 'kappa_gcn_v2']
DATASETS    = ['wordnet_mammals', 'ogbn_arxiv']
ALPHA       = 0.05     # before Bonferroni correction

print('Loading results...')

In [ ]:
def load_seed_results(model, dataset):
    """Load test metrics for all 10 seeds for a given model-dataset pair."""
    results = []
    for seed in SEEDS:
        p = OUTPUT_ROOT / f'{model}_{dataset}_seed{seed}' / 'results.json'
        if p.exists():
            with open(p) as f:
                r = json.load(f)
            results.append(r['test_metrics'])
        else:
            print(f'  WARNING: missing {p}')
    return results

# Collect all results
all_results = {}
for dataset in DATASETS:
    all_results[dataset] = {}
    for model in MODELS:
        all_results[dataset][model] = load_seed_results(model, dataset)
        n = len(all_results[dataset][model])
        print(f'  {model:20s} × {dataset}: {n}/10 seeds loaded')

In [ ]:
def summarise(results_list, metric):
    """Compute mean, std, 95% CI for a metric across seeds."""
    vals = np.array([r[metric] for r in results_list if metric in r])
    if len(vals) == 0:
        return dict(mean=np.nan, std=np.nan, ci95=np.nan, n=0, vals=vals)
    return dict(
        mean=vals.mean(),
        std=vals.std(),
        ci95=1.96 * vals.std() / np.sqrt(len(vals)),
        n=len(vals),
        vals=vals,
    )

def cohen_d(a, b):
    """Cohen's d effect size between two arrays."""
    pooled_std = np.sqrt((a.std()**2 + b.std()**2) / 2)
    return abs(a.mean() - b.mean()) / (pooled_std + 1e-12)

print('Summary functions defined.')

In [ ]:
# ── WordNet-Mammals ────────────────────────────────────────────────────────────
print('\n=== WordNet-Mammals (Table 2, left columns) ===')

dataset = 'wordnet_mammals'
geonet_map  = summarise(all_results[dataset]['geonet'], 'MAP')
geonet_dist = summarise(all_results[dataset]['geonet'], 'distortion')

rows = []
pvalues = []
for model in MODELS:
    r_map  = summarise(all_results[dataset][model], 'MAP')
    r_dist = summarise(all_results[dataset][model], 'distortion')

    # Wilcoxon signed-rank test vs GeoNet
    if model != 'geonet' and len(r_map['vals']) == len(geonet_map['vals']) == 10:
        stat, pval = wilcoxon(geonet_map['vals'], r_map['vals'])
        d = cohen_d(geonet_map['vals'], r_map['vals'])
    else:
        pval, d = np.nan, np.nan

    rows.append({
        'Model':      model,
        'MAP':        f"{r_map['mean']:.3f} ± {r_map['std']:.3f}",
        'Distortion': f"{r_dist['mean']:.3f}",
        'p-value':    pval,
        "Cohen's d":  d,
        'n_seeds':    r_map['n'],
    })
    pvalues.append(pval)

# Bonferroni correction over k=48 comparisons (paper Section 4.2)
k_comparisons = 48
alpha_corrected = ALPHA / k_comparisons
print(f'Bonferroni-corrected alpha: {alpha_corrected:.4f}')

df_wn = pd.DataFrame(rows)
df_wn['Significant*'] = df_wn['p-value'] < alpha_corrected
print(df_wn.to_string(index=False))

In [ ]:
# ── ogbn-arxiv ─────────────────────────────────────────────────────────────────
print('\n=== ogbn-arxiv (Table 2, right columns) ===')

dataset = 'ogbn_arxiv'
geonet_acc = summarise(all_results[dataset]['geonet'], 'accuracy')

rows_ogbn = []
for model in MODELS:
    r_acc  = summarise(all_results[dataset][model], 'accuracy')
    r_dist = summarise(all_results[dataset][model], 'distortion')

    if model != 'geonet' and len(r_acc['vals']) == len(geonet_acc['vals']) == 10:
        _, pval = wilcoxon(geonet_acc['vals'], r_acc['vals'])
        d = cohen_d(geonet_acc['vals'], r_acc['vals'])
    else:
        pval, d = np.nan, np.nan

    rows_ogbn.append({
        'Model':       model,
        'Accuracy (%)': f"{r_acc['mean']*100:.1f} ± {r_acc['std']*100:.1f}",
        'Distortion':   f"{r_dist['mean']:.3f}",
        'p-value':      pval,
        "Cohen's d":    d,
    })

df_ogbn = pd.DataFrame(rows_ogbn)
df_ogbn['Significant*'] = df_ogbn['p-value'] < alpha_corrected
print(df_ogbn.to_string(index=False))

In [ ]:
# ── Save Table 2 to CSV ───────────────────────────────────────────────────────
import os
os.makedirs('outputs/tables', exist_ok=True)
df_wn.to_csv('outputs/tables/table2_wordnet.csv', index=False)
df_ogbn.to_csv('outputs/tables/table2_ogbn.csv',  index=False)
print('Tables saved to outputs/tables/')
print('\nPaper Table 2 reproduction complete.')